# PRABHĀSA — Minimal BLiMP Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/PSALM/blob/main/notebooks/demo_03_reproduce_blimp.ipynb)

Run a minimal version of the BLiMP (Benchmark of Linguistic Minimal Pairs) evaluation suite. This demonstrates how PRABHĀSA evaluates grammatical competence on English structural phenomena using pseudo-log-likelihood (PLL) scoring of minimal pairs.

The full BLiMP suite has ~67,000 pairs across 12 phenomena; this notebook shows a handful of examples from each category to illustrate the method.

In [ ]:
%pip install -q --upgrade transformers torch sentencepiece

In [ ]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer
from collections import defaultdict

MODEL_ID = "qbz506/prabhasa-b_ss-0.1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_ID} on {device}...")

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
).to(device).eval()

print(f"✓ Model loaded.")

In [ ]:
@torch.no_grad()
def pseudo_log_likelihood(text: str) -> float:
    """
    Pseudo-log-likelihood (PLL) score: sum of log p(token | context with that token masked).
    Higher score = model prefers this completion.
    
    This is the standard BLiMP scoring method (Salazar et al., 2020).
    """
    ids = tok(text, return_tensors="pt").input_ids.to(device)
    mask_id = tok.mask_token_id if tok.mask_token_id is not None else tok.vocab_size - 1
    total = 0.0
    token_count = 0

    for i in range(1, ids.size(1) - 1):  # Skip [CLS] and [SEP]-like
        masked = ids.clone()
        gold = ids[0, i].item()
        masked[0, i] = mask_id
        logits = model(masked).logits[0, i]
        total += torch.log_softmax(logits, dim=-1)[gold].item()
        token_count += 1

    return total / max(token_count, 1)  # Normalize by token count


# Define minimal pair examples from major BLiMP phenomena
blimp_examples = {
    "1. Subject-Verb Agreement": [
        ("The cat sleeps on the mat.", "The cat sleep on the mat."),
        ("The cats sleep in the garden.", "The cats sleeps in the garden."),
        ("John has finished his homework.", "John have finished his homework."),
    ],
    "2. Reflexive Binding": [
        ("The boy sees himself in the mirror.", "The boy sees herself in the mirror."),
        ("Mary respects herself.", "Mary respects himself."),
        ("The teachers helped themselves.", "The teachers helped himself."),
    ],
    "3. Determiner-Noun Agreement": [
        ("This cars are parked here.", "These cars are parked here."),
        ("That dog is friendly.", "Those dog is friendly."),
    ],
    "4. Pronoun Case": [
        ("She gave the book to him.", "She gave the book to he."),
        ("Between you and me, this is difficult.", "Between you and I, this is difficult."),
    ],
    "5. Irregular Forms": [
        ("The mouse ran away quickly.", "The mouse runned away quickly."),
        ("She went to the store.", "She goed to the store."),
        ("They ate the cake.", "They eated the cake."),
    ],
    "6. Long-Range Dependencies": [
        (
            "The book that I read last week was interesting.",
            "The book that I readed last week was interesting.",
        ),
        (
            "The managers who conducted the interviews were impressed.",
            "The managers who conducted the interview were impressed.",
        ),
    ],
    "7. Tense Consistency": [
        ("She studies hard and passes the exam.", "She studies hard and passed the exam."),
        ("They walked to the park and played basketball.", "They walked to the park and plays basketball."),
    ],
    "8. Negative Concord": [
        ("The student has no pen.", "The student has not no pen."),
        ("I found nothing.", "I found anything."),
    ],
    "9. Quantity Agreement": [
        ("Many students attended the lecture.", "Much students attended the lecture."),
        ("A little time remains.", "A few time remains."),
    ],
}

print("\n" + "="*80)
print("BLiMP Minimal-Pair Evaluation (Demo)")
print("="*80)
print(f"\nModel: {MODEL_ID}")
print(f"Method: Pseudo-log-likelihood (PLL) scoring")
print(f"Task: Assign higher score to grammatical (good) sentence over ungrammatical (bad)\n")

# Score all pairs
results = {}
for category, pairs in blimp_examples.items():
    category_correct = 0
    category_total = len(pairs)

    print(f"\n{category}")
    print("-" * 80)

    for good, bad in pairs:
        score_good = pseudo_log_likelihood(good)
        score_bad = pseudo_log_likelihood(bad)
        correct = score_good > score_bad
        category_correct += correct

        status = "✓ CORRECT" if correct else "✗ INCORRECT"
        print(f"  {status}")
        print(f"    Good: {score_good:+.4f}  |  {good!r}")
        print(f"    Bad:  {score_bad:+.4f}  |  {bad!r}")
        print()

    accuracy = category_correct / category_total
    results[category] = {"correct": category_correct, "total": category_total, "accuracy": accuracy}
    print(f"  Category accuracy: {category_correct}/{category_total} = {accuracy:.1%}")

## Summary statistics

In [ ]:
import matplotlib.pyplot as plt

# Overall accuracy
total_correct = sum(r["correct"] for r in results.values())
total_pairs = sum(r["total"] for r in results.values())
overall_acc = total_correct / total_pairs

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nOverall accuracy: {total_correct}/{total_pairs} = {overall_acc:.1%}")
print(f"\nPer-category breakdown:")
print(f"{'Category':<40} {'Accuracy':>15}")
print("-" * 60)
for category, result in results.items():
    acc = result["accuracy"]
    print(f"{category:<40} {acc:>14.1%}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart by category
categories = [c.split(".")[1].strip() for c in results.keys()]
accuracies = [results[c]["accuracy"] for c in results.keys()]

axes[0].barh(categories, accuracies, color="steelblue", alpha=0.7, edgecolor="black")
axes[0].set_xlabel("Accuracy", fontsize=12)
axes[0].set_title("BLiMP Phenomena Accuracy (Mini-Suite)", fontsize=12, fontweight="bold")
axes[0].set_xlim([0, 1.0])
axes[0].axvline(x=overall_acc, color="red", linestyle="--", linewidth=2, label=f"Overall: {overall_acc:.1%}")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.3)

# Aggregate pie chart
axes[1].pie(
    [total_correct, total_pairs - total_correct],
    labels=[f"Correct\n({total_correct})", f"Incorrect\n({total_pairs - total_correct})"],
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90,
)
axes[1].set_title(f"Overall Performance\n({total_pairs} pairs)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## Full BLiMP

This notebook runs a **minimal demo** (27 pairs across 9 phenomena). The official BLiMP suite contains:

- **~67,000 minimal pairs** across 12 phenomena:
  1. Agreement violations
  2. Anaphor agreement
  3. Binding
  4. Control/Raising
  5. Determiner-Noun Agreement
  6. Ellipsis
  7. Filler-Gap Dependencies
  8. Irregular Forms
  9. Island Effects
  10. NPI Licensing
  11. Quantifiers
  12. Subject-Verb Agreement

To reproduce the full results from the paper, use the official evaluation script in the repository:

```bash
git clone https://github.com/SharathSPhD/PSALM.git
cd PSALM
pip install -e . sentencepiece
python scripts/official_eval.py --ckpt <path-to-checkpoint> --name <run-name>
```

This will produce:
- Per-phenomenon accuracies
- An aggregate BLiMP score (average across all 12 phenomena)
- Comparison to COGS, CFQ, SCAN, and other structural generalization benchmarks

See the [results page](https://SharathSPhD.github.io/PSALM/results/) for the published numbers.